In [ ]:
import numpy as np
import xarray as xr
from glob import glob
import random

import os
from tqdm.notebook import tqdm
import re

import pop_tools

%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib

import dask

import string
import box_model

import PyCO2SYS as pyco2
from PyCO2SYS import CO2SYS

In [ ]:
def calcCO2SYS(alk,    # (in umol/kg)
               dic,    # (in umol/kg)
               sal      =   35,  # Salinity of the sample
               temp     =   10,  # Temperature at input conditions
               sil      =   0.5,  # Concentration of silicate  in the sample (in umol/kg)
               po4      =    2,  # Concentration of phosphate in the sample (in umol/kg)
               pres     =    0,  # Pressure    at input conditions
               pHscale  =    1,  # pH scale at which the input pH is reported ("1" means "Total Scale")
               k1k2c    =    4,  # Choice of H2CO3 and HCO3- dissociation constants K1 and K2 ("4" means "Mehrbach refit")
               kso4c    =    1  # Choice of HSO4- dissociation constants KSO4 ("1" means "Dickson"
  ):
  delta = 0.1 # µmol/L or mmol/m^3
  csys = CO2SYS(alk,dic,1,2,sal,temp,temp,pres,pres,sil,po4, pHscale, k1k2c, kso4c)
  csys_dic = CO2SYS(alk,dic+delta,1,2,sal,temp,temp,pres,pres,sil,po4, pHscale, k1k2c, kso4c)
  csys_alk = CO2SYS(alk+delta,dic,1,2,sal,temp,temp,pres,pres,sil,po4, pHscale, k1k2c, kso4c)

  dpCO2dDIC = (csys_dic['pCO2in'] - csys['pCO2in'])/delta
  dpCO2dAlk = (csys_alk['pCO2in'] - csys['pCO2in'])/delta
  dCO2dDIC = (csys_dic['CO2out'] - csys['CO2out'])/delta
  dCO2dAlk = (csys_alk['CO2out'] - csys['CO2out'])/delta
  eta_co2 = -dpCO2dAlk/dpCO2dDIC
  csys["dpCO2dDIC"] = dpCO2dDIC
  csys["dpCO2dAlk"] = dpCO2dAlk
  csys["dCO2dDIC"] = dCO2dDIC
  csys["dCO2dAlk"] = dCO2dAlk
  csys["eta_co2"] = eta_co2
  csys["dDICdAlk"] = eta_co2

  if(type(alk)==np.ndarray):
    for k in csys.keys():
      csys[k] = np.reshape(csys[k], newshape=alk.shape)

  return csys

## Figure 1

In [ ]:
ocn_state = {
    "dic": 1960+10,  # mmol/m^3
    "alk": 2260+7,  # mmol/m^3
    "salt": 34,  #
    "temp": 24,  # °C
    "sio3": 4.23,  # mmol/m^3
    "po4": 0.12,  # mmol/m^3
}
ocn_state

In [ ]:
%%time
dic_range = np.arange(1900, 2060, 20)
alk_range = np.arange(2200, 2380, 20)
ALK, DIC = np.meshgrid(alk_range, dic_range)

### ALk vs. DIC in ocean
SSS = ocn_state["salt"] * np.ones(DIC.shape)
SST = ocn_state["temp"] * np.ones(DIC.shape)

csys_solver_space = box_model.calc_csys(DIC, ALK, SSS, SST)

# set initial state
alk_start = ocn_state["alk"]
dic_start = ocn_state["dic"]

csys_solver_start = box_model.calc_csys(
    dic_start,
    alk_start,
    ocn_state["salt"],
    ocn_state["temp"],
)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

cs = ax.contour(ALK, DIC, csys_solver_space.pco2,
    colors="k", linewidths=1.0,
)
ax.clabel(cs, cs.levels, inline=True, fontsize=10)

cs = ax.contour(ALK, DIC, csys_solver_space.pco2, levels=[csys_solver_start.pco2], colors="k", linewidths=1.0, linestyles="--",)
ax.clabel(cs, fmt='%0.0f', inline=True, fontsize=10)
print(csys_solver_start.pco2)

head_length = 4

# set alkalinity addition and compute end state

intervention_size = 55
######## OAE
d_alk = intervention_size
d_dic = 0

alk_end = alk_start + d_alk
dic_end = csys_solver_start.calc_new_dic_w_oae(alk_end)

csys_solver_end = box_model.calc_csys(
    dic_end, alk_end, ocn_state["salt"], ocn_state["temp"]
)
csys_solver_mid = box_model.calc_csys(
    dic_start, alk_end, ocn_state["salt"], ocn_state["temp"]
)
cs = ax.contour(ALK, DIC, csys_solver_space.pco2, levels=[csys_solver_mid.pco2], colors="tab:blue", linewidths=1.0, linestyles="--")
ax.clabel(cs, fmt='%0.0f', inline=True, fontsize=10)
print(csys_solver_mid.pco2)

ax.text(
    np.mean([alk_start, alk_end])+3,
    dic_start + np.diff(ax.get_ylim()) * 0.02,
    "OAE",
    ha="center",
    fontsize=12,
    fontweight="bold",
    color="tab:blue",
    backgroundcolor="w",
    zorder=100,
)

# determine equivalent DOR
dor_delta_dic = dic_end - (dic_start + d_dic)
ax.arrow(alk_start, dic_start, d_alk - head_length, 0, head_width=3, head_length=head_length, linewidth=1, color="tab:blue", zorder=1000,)
ax.arrow(alk_start + d_alk, dic_start + d_dic, 0, dor_delta_dic - head_length, head_width=3, head_length=head_length, linewidth=2.7, color="tab:blue", zorder=1000,)

ax.plot(alk_start, dic_start, "ko", zorder=100)
ax.plot(alk_end, dic_end, "ko", zorder=100)

######## DOR
dor_delta_dic = intervention_size
ax.arrow(alk_start, dic_start, 0, - (dor_delta_dic - head_length), head_width=4, head_length=head_length, linewidth=0, color="tab:red", zorder=1000,)
#ax.arrow(alk_start+2, dic_start - dor_delta_dic + 2, 0, dor_delta_dic - head_length, head_width=3,head_length=head_length, color="tab:red", zorder=1000,)
ax.arrow(alk_start, dic_start - dor_delta_dic, 0, dor_delta_dic - head_length, head_width=5, head_length=head_length+1, linewidth=0, color="tab:red", zorder=1000,)
ax.text(
    alk_start + np.diff(ax.get_xlim()) * 0.03 - 12,
    np.mean([dic_start, dic_start - (dic_end - dic_start)])-3,
    "DOR",
    ha="center",
    va="center",
    rotation=90,
    fontsize=12,
    fontweight="bold",
    color="tab:red",
    backgroundcolor="w",
)

ax.plot([alk_start-1, alk_start-1], [dic_start, dic_start-dor_delta_dic+2.5], 'r', linewidth=1)
ax.plot([alk_start+0.7, alk_start+0.7], [dic_start, dic_start-dor_delta_dic+2.5], 'r', linewidth=2.5)

csys_solver_mid_dor = box_model.calc_csys(
    dic_start-dor_delta_dic, alk_start, ocn_state["salt"], ocn_state["temp"]
)
cs = ax.contour(ALK, DIC, csys_solver_space.pco2, levels=[csys_solver_mid_dor.pco2], colors="tab:red", linewidths=1.0, linestyles="--")
ax.clabel(cs, fmt='%0.0f', inline=True, fontsize=10)
print(csys_solver_mid_dor.pco2)

######## ACE
d_alk = intervention_size
d_dic = intervention_size

alk_end = alk_start + d_alk
dic_end = csys_solver_start.calc_new_dic_w_oae(alk_end)

csys_solver_end = box_model.calc_csys(
    dic_end, alk_end, ocn_state["salt"], ocn_state["temp"]
)
csys_solver_mid = box_model.calc_csys(
    dic_start + d_dic, alk_end, ocn_state["salt"], ocn_state["temp"]
)
cs = ax.contour(ALK, DIC, csys_solver_space.pco2, levels=[csys_solver_mid.pco2], colors="tab:purple", linewidths=1.0, linestyles="--")
labels = ax.clabel(cs, fmt='%0.0f', inline=True, fontsize=10)
print(csys_solver_mid.pco2)

# determine equivalent DOR
dor_delta_dic = dic_end - (dic_start + d_dic)
ax.arrow(alk_start, dic_start, d_alk - head_length/2**0.5, d_dic - head_length/2**0.5, head_width=3, head_length=head_length, linewidth=1, color="tab:purple", zorder=1000,)
ax.arrow(alk_start + d_alk, dic_start + d_dic-1, 0, -np.abs(dor_delta_dic + head_length)+2, head_width=3,head_length=head_length, linewidth=2.7, color="tab:purple", zorder=1000, )

ax.text(
    np.mean([alk_start, alk_end])-5,
    np.mean([dic_start, dic_end]),
    "ACE",
    ha="center",
    fontsize=12,
    fontweight="bold",
    color="tab:purple",
    rotation=45,
    backgroundcolor="w",
    zorder=100,
)

ax.plot(alk_start, dic_start, "ko", zorder=100, markersize=7)
ax.plot(alk_end, dic_end, "ko", zorder=100, markersize=7)

ax.text(alk_start - np.diff(ax.get_xlim()) * 0.02-4, dic_start-4, "Initial\nstate", ha="right", fontsize=12, fontweight="bold", backgroundcolor="w")

ax.set_ylabel("DIC [mmol m$^{-3}$]")
ax.set_xlabel("Alk [mmol m$^{-3}$]")
ax.set_xlim(2225, 2360)
ax.set_ylim(1910, 2030)


fig.savefig('./figures/Figure_1.png', dpi=200, bbox_inches='tight')

## Figure 6 part1

In [ ]:
def plot_arrows(ax, dic_range, alk_range):

    intervention_size = 150

    arrow_width = 8
    head_length = 6

    # dic_range = np.arange(1900, 2300, 10)
    # alk_range = np.arange(2200, 2600, 10)
    ALK, DIC = np.meshgrid(alk_range, dic_range)
    
    ### ALk vs. DIC in ocean
    SSS = ocn_state["salt"] * np.ones(DIC.shape)
    SST = ocn_state["temp"] * np.ones(DIC.shape)
    
    csys_solver_space = box_model.calc_csys(DIC, ALK, SSS, SST)

    alk_start = 2330
    dic_start = 2070
    
    csys_solver_start = box_model.calc_csys(
        dic_start,
        alk_start,
        ocn_state["salt"],
        ocn_state["temp"],
    )
        
    cs = ax.contour(ALK, DIC, csys_solver_space.pco2, levels=[csys_solver_start.pco2], colors="k", linewidths=1.0, linestyles="--",)
    ax.clabel(cs, fmt='%0.0f', inline=True, fontsize=10)

    print(csys_solver_start.pco2)
    
    # set alkalinity addition and compute end state
    
    ######## OAE
    d_alk = intervention_size
    d_dic = 0
    
    alk_end = alk_start + d_alk
    dic_end = csys_solver_start.calc_new_dic_w_oae(alk_end)
    
    csys_solver_end = box_model.calc_csys(
        dic_end, alk_end, ocn_state["salt"], ocn_state["temp"]
    )
    csys_solver_mid = box_model.calc_csys(
        dic_start, alk_end, ocn_state["salt"], ocn_state["temp"]
    )
    cs = ax.contour(ALK, DIC, csys_solver_space.pco2, levels=[csys_solver_mid.pco2], colors="tab:blue", linewidths=1.0, linestyles="--")
    ax.clabel(cs, fmt='%0.0f', inline=True, fontsize=10)
    
    ax.text(
        np.mean([alk_start, alk_end])-3,
        dic_start + np.diff(ax.get_ylim()) * 0.02+5,
        "OAE",
        ha="center",
        fontsize=11,
        fontweight="bold",
        color="tab:blue",
        #backgroundcolor='w',
        zorder=100,
    )
    
    # determine equivalent DOR
    dor_delta_dic = dic_end - (dic_start + d_dic)
    ax.arrow(alk_start, dic_start, d_alk - head_length, 0, head_width=arrow_width, head_length=head_length, linewidth=1, color="tab:blue", zorder=1000,)
    ax.arrow(alk_start + d_alk, dic_start + d_dic, 0, dor_delta_dic - head_length, head_width=arrow_width, head_length=head_length, linewidth=2.7, color="tab:blue", zorder=1000,)
    
    ax.plot(alk_start, dic_start, "ko", zorder=100)
    ax.plot(alk_end, dic_end, "ko", zorder=100)
    
    ######## DOR
    dor_delta_dic = intervention_size # force it change to 50
    ax.arrow(alk_start, dic_start, 0, - (dor_delta_dic - head_length), head_width=arrow_width, head_length=head_length, linewidth=1, color="tab:red", zorder=1000,)
    #ax.arrow(alk_start+2, dic_start - dor_delta_dic + 2, 0, dor_delta_dic - head_length, head_width=3,head_length=head_length, color="tab:red", zorder=1000,)
    ax.arrow(alk_start, dic_start - dor_delta_dic, 0, dor_delta_dic - head_length, head_width=arrow_width, head_length=head_length+1, linewidth=1, color="tab:red", zorder=1000,)
    ax.text(
        alk_start + np.diff(ax.get_xlim()) * 0.03 - 30,
        np.mean([dic_start, dic_start - (dic_end - dic_start)]),
        "DOR",
        ha="center",
        va="center",
        rotation=90,
        fontsize=11,
        fontweight="bold",
        color="tab:red",
        #backgroundcolor='w',
    )
    
    ax.plot([alk_start-1, alk_start-1], [dic_start, dic_start-dor_delta_dic+2.5], 'r', linewidth=1)
    ax.plot([alk_start+0.7, alk_start+0.7], [dic_start, dic_start-dor_delta_dic+2.5], 'r', linewidth=2.5)
    #ax.plot([alk_start, alk_start], [dic_start, dic_start-dor_delta_dic+1], 'r')
    
    csys_solver_mid_dor = box_model.calc_csys(
        dic_start-dor_delta_dic, alk_start, ocn_state["salt"], ocn_state["temp"]
    )
    cs = ax.contour(ALK, DIC, csys_solver_space.pco2, levels=[csys_solver_mid_dor.pco2], colors="tab:red", linewidths=1.0, linestyles="--")
    ax.clabel(cs, fmt='%0.0f', inline=True, fontsize=10)
    print(csys_solver_mid_dor.pco2)
    
    ######## ACE
    d_alk = intervention_size
    d_dic = intervention_size
    
    alk_end = alk_start + d_alk
    dic_end = csys_solver_start.calc_new_dic_w_oae(alk_end)
    
    csys_solver_end = box_model.calc_csys(
        dic_end, alk_end, ocn_state["salt"], ocn_state["temp"]
    )
    csys_solver_mid = box_model.calc_csys(
        dic_start + d_dic, alk_end, ocn_state["salt"], ocn_state["temp"]
    )
    cs = ax.contour(ALK, DIC, csys_solver_space.pco2, levels=[csys_solver_mid.pco2], colors="tab:purple", linewidths=1.0, linestyles="--")
    ax.clabel(cs, fmt='%0.0f', inline=True, fontsize=10)
    
    # determine equivalent DOR
    dor_delta_dic = dic_end - (dic_start + d_dic)
    ax.arrow(alk_start, dic_start, d_alk - head_length/2**0.5, d_dic - head_length/2**0.5, head_width=arrow_width, head_length=head_length, linewidth=1, color="tab:purple", zorder=1000,)
    ax.arrow(alk_start + d_alk, dic_start + d_dic-1, 0, -np.abs(dor_delta_dic + head_length)+2, head_width=arrow_width,head_length=head_length, linewidth=2.7, color="tab:purple", zorder=1000, )
    
    ax.text(
        np.mean([alk_start, alk_end])-35,
        np.mean([dic_start, dic_end])+5,
        "ACE",
        ha="center",
        fontsize=11,
        fontweight="bold",
        color="tab:purple",
        rotation=45,
        #backgroundcolor='w',
        zorder=100,
    )
    
    
    ax.plot(alk_start, dic_start, "ko", zorder=100, markersize=7)
    ax.plot(alk_end, dic_end, "ko", zorder=100, markersize=7)

In [ ]:
def calcCO2SYS(alk,    # (in umol/kg)
               dic,    # (in umol/kg)
               sal      =   35,  # Salinity of the sample
               temp     =   10,  # Temperature at input conditions
               sil      =   0.5,  # Concentration of silicate  in the sample (in umol/kg)
               po4      =    2,  # Concentration of phosphate in the sample (in umol/kg)
               pres     =    0,  # Pressure    at input conditions
               pHscale  =    1,  # pH scale at which the input pH is reported ("1" means "Total Scale")
               k1k2c    =    4,  # Choice of H2CO3 and HCO3- dissociation constants K1 and K2 ("4" means "Mehrbach refit")
               kso4c    =    1  # Choice of HSO4- dissociation constants KSO4 ("1" means "Dickson"
  ):
  delta = 0.1 # µmol/L or mmol/m^3
  csys = CO2SYS(alk,dic,1,2,sal,temp,temp,pres,pres,sil,po4, pHscale, k1k2c, kso4c)
  csys_dic = CO2SYS(alk,dic+delta,1,2,sal,temp,temp,pres,pres,sil,po4, pHscale, k1k2c, kso4c)

  dCO2dDIC = (csys_dic['CO2out'] - csys['CO2out'])/delta
  csys["dCO2dDIC"] = dCO2dDIC

  if(type(alk)==np.ndarray):
    for k in csys.keys():
      csys[k] = np.reshape(csys[k], shape=alk.shape)

  return csys

In [ ]:
%%time
plt.rcParams.update({'font.size': 12})

dic_range = np.arange(1900, 2300, 10)
alk_range = np.arange(2200, 2600, 10)
ALK, DIC = np.meshgrid(alk_range, dic_range)
csys = calcCO2SYS(ALK, DIC)

eta_max = 1/csys["isoQout"]
tau = 1/csys["dCO2dDIC"]


fig, axs = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={'width_ratios': [1, 1]})
# Left subplot
cf1 = axs[0].contourf(ALK, DIC, eta_max, levels=8, alpha=0.5, cmap='GnBu')
cbar1 = fig.colorbar(cf1, ax=axs[0])
#cbar1.set_label(r'% (mmol m$^{-3}$)$^{-1}$')
cbar1.ax.set_title(rf'$\eta_{{\mathrm{{max}}}}$', pad=10)  # pad moves it upward


axs[0].set_xlabel(r'Alkalinity [mmol m$^{-3}$]')
axs[0].set_ylabel(r'DIC [mmol m$^{-3}$]')

# Right subplot 
cf2 = axs[1].contourf(ALK, DIC, tau, levels=8, alpha=0.5, cmap='GnBu')
cbar2 = fig.colorbar(cf2, ax=axs[1])
#cbar2.ax.set_title(r'$\frac{\partial \mathrm{DIC}}{\partial \mathrm{CO_2}}$', pad=10)
cbar2.ax.set_title(r'$\beta$', pad=10)


axs[1].set_xlabel(r'Alkalinity [mmol m$^{-3}$]')
axs[1].set_ylabel(r'DIC [mmol m$^{-3}$]')

axs[0].text(-0.15, 1.05, 'a', transform=axs[0].transAxes, fontsize=16, fontweight='bold', va='top')
axs[1].text(-0.15, 1.05, 'b', transform=axs[1].transAxes, fontsize=16, fontweight='bold', va='top')


plot_arrows(axs[0], dic_range, alk_range)
plot_arrows(axs[1], dic_range, alk_range)

fig.tight_layout()
fig.savefig('./figures/Figure_6_part1.png', dpi=200, bbox_inches='tight')

## Figure S10
Percent of nonlinearity per unit of intervention

In [ ]:
def cal_nonlinearity(ALK, DIC, interventions, delta_alk=1, delta_dic=0):

    eta_max_all = np.empty((interventions.shape[0], ALK.shape[0], ALK.shape[1]))
    tau_all = np.empty((interventions.shape[0], ALK.shape[0], ALK.shape[1]))
    
    for i in range(len(interventions)):
    
        csys_ = calcCO2SYS(ALK + interventions[i] * delta_alk, DIC + interventions[i] * delta_dic)
        eta_max_all[i,:,:] = 1/csys_["isoQout"]
        tau_all[i,:,:] = 1/csys_["dCO2dDIC"]
    
    return eta_max_all, tau_all

def comp_percent(eta_max_all, tau_all, interventions):
    
    eta_max_percent_increase = (eta_max_all[-1,:,:] - eta_max_all[0,:,:]) / eta_max_all[0,:,:] / (interventions[-1] - interventions[0]) * 100 # % increase per µM of intervention
    tau_percent_increase = (tau_all[-1,:,:] - tau_all[0,:,:]) / tau_all[0,:,:] / (interventions[-1] - interventions[0]) * 100

    return eta_max_percent_increase, tau_percent_increase

In [ ]:
%%time
dic_range = np.arange(1900, 2300, 10)
alk_range = np.arange(2200, 2600, 8)
ALK, DIC = np.meshgrid(alk_range, dic_range)
interventions = np.arange(0,201,10)

eta_max_all_oae, tau_all_oae = cal_nonlinearity(ALK, DIC, interventions, delta_alk=1, delta_dic=0)
eta_max_all_dor, tau_all_dor = cal_nonlinearity(ALK, DIC, interventions, delta_alk=0, delta_dic=-1)
eta_max_all_erw, tau_all_erw = cal_nonlinearity(ALK, DIC, interventions, delta_alk=1, delta_dic=1)

eta_max_nonlin_oae, tau_nonlin_oae = comp_percent(eta_max_all_oae, tau_all_oae, interventions)
eta_max_nonlin_dor, tau_nonlin_dor = comp_percent(eta_max_all_dor, tau_all_dor, interventions)
eta_max_nonlin_erw, tau_nonlin_erw = comp_percent(eta_max_all_erw, tau_all_erw, interventions)

In [ ]:
plt.rcParams.update({'font.size': 10})

fig, axs = plt.subplots(3, 2, figsize=(9, 8), gridspec_kw={'width_ratios': [1, 1]})
axs = axs.flatten()

##### OAE
# Left subplot
cf1 = axs[0].contourf(ALK, DIC, eta_max_nonlin_oae, levels=8, cmap='viridis_r')
cbar1 = fig.colorbar(cf1, ax=axs[0])
cbar1.set_label(r'% (mmol m$^{-3}$)$^{-1}$')
cbar1.ax.invert_yaxis()
cbar1.ax.set_title(rf'$\eta_{{\mathrm{{max}}}}$', pad=10)  # pad moves it upward


axs[0].set_xlabel(r'Alkalinity [mmol m$^{-3}$]')
axs[0].set_ylabel(r'DIC [mmol m$^{-3}$]')

# Right subplot 
cf2 = axs[1].contourf(ALK, DIC, tau_nonlin_oae, levels=8, cmap='viridis')
cbar2 = fig.colorbar(cf2, ax=axs[1])
cbar2.set_label(r'% (mmol m$^{-3}$)$^{-1}$')
cbar2.ax.set_title(rf'$\beta$', pad=10)  # pad moves it upward

axs[1].set_xlabel(r'Alkalinity [mmol m$^{-3}$]')
axs[1].set_ylabel(r'DIC [mmol m$^{-3}$]')

axs[0].text(-0.2, 1.05, 'a', transform=axs[0].transAxes, fontsize=13, fontweight='bold', va='top')
axs[1].text(-0.2, 1.05, 'b', transform=axs[1].transAxes, fontsize=13, fontweight='bold', va='top')


######### DOR
# Left subplot
cf1 = axs[2].contourf(ALK, DIC, eta_max_nonlin_dor, levels=8, cmap='viridis_r')
cbar1 = fig.colorbar(cf1, ax=axs[2])
cbar1.set_label(r'% (mmol m$^{-3}$)$^{-1}$')
cbar1.ax.invert_yaxis()

axs[2].set_xlabel(r'Alkalinity [mmol m$^{-3}$]')
axs[2].set_ylabel(r'DIC [mmol m$^{-3}$]')

# Right subplot 
cf2 = axs[3].contourf(ALK, DIC, tau_nonlin_dor, levels=8, cmap='viridis')
cbar2 = fig.colorbar(cf2, ax=axs[3])
cbar2.set_label(r'% (mmol m$^{-3}$)$^{-1}$')

axs[3].set_xlabel(r'Alkalinity [mmol m$^{-3}$]')
axs[3].set_ylabel(r'DIC [mmol m$^{-3}$]')

axs[2].text(-0.2, 1.05, 'c', transform=axs[2].transAxes, fontsize=13, fontweight='bold', va='top')
axs[3].text(-0.2, 1.05, 'd', transform=axs[3].transAxes, fontsize=13, fontweight='bold', va='top')


######## ERW
# Left subplot
cf1 = axs[4].contourf(ALK, DIC, eta_max_nonlin_erw, levels=8, cmap='viridis')
cbar1 = fig.colorbar(cf1, ax=axs[4])
cbar1.set_label(r'% (mmol m$^{-3}$)$^{-1}$')


axs[4].set_xlabel(r'Alkalinity [mmol m$^{-3}$]')
axs[4].set_ylabel(r'DIC [mmol m$^{-3}$]')

# Right subplot 
cf2 = axs[5].contourf(ALK, DIC, tau_nonlin_erw, levels=8, cmap='viridis_r')
cbar2 = fig.colorbar(cf2, ax=axs[5])
cbar2.set_label(r'% (mmol m$^{-3}$)$^{-1}$')
cbar2.ax.invert_yaxis()

axs[5].set_xlabel(r'Alkalinity [mmol m$^{-3}$]')
axs[5].set_ylabel(r'DIC [mmol m$^{-3}$]')


axs[4].text(-0.2, 1.05, 'e', transform=axs[4].transAxes, fontsize=13, fontweight='bold', va='top')
axs[5].text(-0.2, 1.05, 'f', transform=axs[5].transAxes, fontsize=13, fontweight='bold', va='top')


axs[0].text(-0.4, 0.5, 'OAE', transform=axs[0].transAxes,
            fontsize=13, 
            va='center', ha='center', rotation='vertical')

axs[2].text(-0.4, 0.5, 'DOR', transform=axs[2].transAxes,
            fontsize=13, 
            va='center', ha='center', rotation='vertical')

axs[4].text(-0.4, 0.5, 'ACE', transform=axs[4].transAxes,
            fontsize=13, 
            va='center', ha='center', rotation='vertical')

fig.tight_layout()

fig.savefig('./figures/Figure_S10.png', dpi=100, bbox_inches='tight')

## Figure 6 part 2

In [ ]:
ocn_state = {
    "dic": 1958.72553055,  # mmol/m^3
    "alk": 2261.89,  # mmol/m^3
    "salt": 34.51,  #
    "temp": 24.37,  # °C
    "sio3": 4.23,  # mmol/m^3
    "po4": 0.12,  # mmol/m^3
}

forcing = dict(
    h=100.0,  # m; mixed layer depth
    u10=5.0,  # m/s; wind speed at 10 m
    Xco2atm=425.0,  # csys_solver.pco2, # µatm; atmospheric CO2
)

bc = dict(dic_bc=2000.0, alk_bc=2250.0)

dic = ocn_state["dic"]
alk = ocn_state["alk"]

ocn_other_state = {k: v for k, v in ocn_state.items() if k not in ["dic", "alk"]}

nday = 365*2

def cal_equilibration(alk, dic, dalk, ddic):

    model = box_model.mixed_layer(
        dic=dic,
        alk=alk,
        tau_dilution=0.0,
        **ocn_other_state,
        **forcing,
        **bc,
        csys_solver="ocmip",
    )
    
    conc_to_m2 = 100/1000
    
    ds_ocmip = model.run(nday, spinup=True, dalk=dalk, ddic=ddic)
    oae_eff = (ds_ocmip.fgco2_cumulative/max(abs(dalk), abs(ddic))/conc_to_m2)

    return oae_eff

In [ ]:
dic_range = np.arange(1900, 3300, 10)
alk_range = np.arange(2200, 2600, 10)
ALK, DIC = np.meshgrid(alk_range, dic_range)
dic_range.shape, alk_range.shape, ALK.shape, DIC.shape

In [ ]:
%%time
interventions = np.arange(0,501,20)
interventions = np.sort(np.append(interventions, 10))
dic_range = [2070]
alk_range = [2330]

ALK, DIC = np.meshgrid(alk_range, dic_range)

num_interv, num_dic, num_alk = len(interventions), len(dic_range), len(alk_range)

true_eta_erw_all = np.empty((num_interv, num_dic, num_alk, nday+1))
appr_eta_erw_all = np.empty((num_interv, num_dic, num_alk, nday+1))

for i in range(num_interv):
    if i%10==0: print(i)
    for j in range(num_dic):
        for k in range(num_alk):

            true_eta_erw_all[i,j,k,:] = 1 + cal_equilibration(ALK[j,k], DIC[j,k], interventions[i], interventions[i])

            true_oae = cal_equilibration(ALK[j,k], DIC[j,k], interventions[i], 0)
            true_dor = cal_equilibration(ALK[j,k], DIC[j,k], 0, -interventions[i])
            appr_eta_erw_all[i,j,k,:] = 1 + true_oae - true_dor
            
ds_erw_curves_phase = xr.Dataset(
    {
        "true_eta_erw": (("intervention", "DIC", "ALK", "time"), true_eta_erw_all),
        "appr_eta_erw": (("intervention", "DIC", "ALK", "time"), appr_eta_erw_all),
    },
    coords={
        "intervention": interventions,
        "ALK": alk_range,
        "DIC": dic_range,
        "time": np.arange(nday+1),
    }
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams.update({'font.size': 12})

time = ds_erw_curves_phase.time.values/30 # month

interv_sel = [100, 200, 500]
ds_sel = ds_erw_curves_phase.sel(intervention=interv_sel)

# make a colormap for interventions
cmap = plt.cm.rainbow
#colors = cmap(np.linspace(0, 1, len(ds_sel.intervention.values)))
colors = ['C0', 'cyan', 'red']

fig, axs = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={'wspace': 0.4})
ax = axs[0]

interv_sel = [10, 40, 100, 200, 500]

for i in range(len(ds_sel.intervention)):

    true_sel = ds_sel.sel(ALK=alk_range[0], DIC=dic_range[0]).isel(intervention=i).true_eta_erw.values - 1
    appr_sel = ds_sel.sel(ALK=alk_range[0], DIC=dic_range[0]).isel(intervention=i).appr_eta_erw.values - 1
    
    ax.plot(time, true_sel, color=colors[i], linestyle='-')
    ax.plot(time, appr_sel, color=colors[i], linestyle='dashed')

ax.set_xlabel('Time (months)')
ax.set_ylabel(rf'$\gamma_{{\mathrm{{ACE}}}}$')


time_steps= [5,20]
for t in time_steps:
    ax.axvline(x=t, color='gray', linestyle='dotted', linewidth=1.5)

# --- Legend 1: interventions ---
interv_handles = [
    Line2D([0], [0], color=colors[i], linestyle='-', label=f'{ds_sel.intervention.values[i]}')
    for i in range(len(ds_sel.intervention))
]
legend1 = ax.legend(
    handles=interv_handles,
    title='Intervention [mmol m$^{-3}$]',
    title_fontsize = 10,
    fontsize=10,
    loc='upper right',
    frameon=False
)

# --- Legend 2: line styles ---
style_handles = [
    Line2D([0], [0], color='k', linestyle='-', label='True'),
    Line2D([0], [0], color='k', linestyle='dashed', label='Approx.')
]
legend2 = ax.legend(
    handles=style_handles,
    title=' ',
    title_fontsize = 10,
    fontsize=10,
    loc='upper left',
    bbox_to_anchor=(0.18, 0.985),  # shift second box to the right
    frameon=False
)

# Add both legends to the same axis
ax.add_artist(legend1)
ax.add_artist(legend2)


#### 2nd panel
ax1 = axs[1]
markers = ['P', 'x', 'd']
for t in range(len(time_steps)):
    t36 = ds_erw_curves_phase.sel(time=time_steps[t]*30, ALK=alk_range[0], DIC=dic_range[0])
    ax1.plot(ds_erw_curves_phase.intervention[::2], ((-t36.appr_eta_erw + t36.true_eta_erw)/(1-t36.true_eta_erw)*100)[::2], marker=markers[t], \
             markersize=6, color='k', label=f't = {(time_steps[t]):.0f} month')

ax1.legend(fontsize=10)
ax1.set_xlabel('Intervention size [mmol m$^{-3}$]')
ax1.set_ylabel('Error in the outgassing frac. (%)')
#ax1.set_ylim(-2,7.5)
#ax1.set_yticks(np.arange(-2,7.5))
ax1.grid(True, which='both', axis='both', linestyle='--', linewidth=0.7, alpha=0.7)

axs[0].text(-0.15, 1.1, 'c', transform=axs[0].transAxes, fontsize=16, fontweight='bold', va='top')
axs[1].text(-0.15, 1.1, 'd', transform=axs[1].transAxes, fontsize=16, fontweight='bold', va='top')


fig.tight_layout()
plt.show()

fig.savefig('./figures/Figure_6_part2.png', dpi=200, bbox_inches='tight')

In [ ]:
from PIL import Image

# Open the two images
fig1 = Image.open('./figures/Figure_6_part1.png')
fig2 = Image.open('./figures/Figure_6_part2.png')

# Get the widths and heights
w1, h1 = fig1.size
w2, h2 = fig2.size

# Create a new image with combined height and max width
new_img = Image.new('RGB', (max(w1, w2), h1 + h2), (255, 255, 255))

# Paste the two images
new_img.paste(fig1, (0, 0))
new_img.paste(fig2, (0, h1))

# Save the combined image
new_img.save('./figures/Figure_6.png', dpi=(200, 200))

plt.imshow(new_img)
